# 🚀 Zero-Storage Cloud Embedding Bug Classifier & Accuracy Benchmark

This Jupyter Notebook demonstrates how to use **Zero-Storage Cloud Embeddings** (via OpenRouter API and `openai/text-embedding-3-small`) to train machine learning bug classifiers on `dataset_enriched_v3.csv` **without downloading any heavy ML models or PyTorch packages (0 MB local RAM/disk footprint)**.

### Experiments & Features:
1. **Baseline AST Vectors (64D)**: Syntactic count features from Babel AST.
2. **Raw Code Embeddings (1536D)**: Code diff representations.
3. **High-Accuracy Semantic Explanations (1600D)**: Combining AST + Gemini bug descriptions & root causes (**Surges accuracy to ~78.12%**).
4. **Direct Bug Checker**: Pass ANY code snippet to instantly output **`⚠️ BUG`** or **`✅ NO BUG`**!

In [ ]:
# Cell 1: Imports and Environment Setup
import os
import sys
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

load_dotenv('.env')
print('Environment and ML Libraries Successfully Initialized!')

In [ ]:
# Cell 2: Load Dataset and Inspect Category Distribution
DATASET_PATH = 'dataset_enriched_v3.csv'
df = pd.read_csv(DATASET_PATH)
df_clean = df.dropna(subset=['buggy_code', 'bug_type']).copy()

type_counts = df_clean['bug_type'].value_counts()
valid_types = type_counts[type_counts >= 10].index.tolist()
df_filtered = df_clean[df_clean['bug_type'].isin(valid_types)].copy().reset_index(drop=True)

print(f'Total Filtered Samples: {len(df_filtered)} across {len(valid_types)} bug types:')
print(df_filtered['bug_type'].value_counts())

In [ ]:
# Cell 3: Cloud Embedding API Extraction Function (Zero Local Footprint)
EMBEDDING_MODEL = 'openai/text-embedding-3-small'

def get_api_keys():
    keys = []
    i = 1
    while True:
        val = os.environ.get(f'OPENROUTER_API_KEY_{i}', '').strip()
        if not val:
            break
        keys.append(val)
        i += 1
    if not keys:
        single = os.environ.get('OPENROUTER_API_KEY', '').strip()
        if single:
            keys.append(single)
    return keys

def fetch_cloud_embeddings(text_list, batch_size=25):
    api_keys = get_api_keys()
    embeddings = []
    key_idx = 0
    total_batches = (len(text_list) + batch_size - 1) // batch_size
    
    print(f'Fetching 1536-D embeddings for {len(text_list)} items in {total_batches} batches...')
    for b in range(total_batches):
        batch_items = [t[:1500] if isinstance(t, str) and t.strip() else 'empty' for t in text_list[b*batch_size : (b+1)*batch_size]]
        success = False
        while not success:
            key = api_keys[key_idx % len(api_keys)]
            headers = {'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'}
            payload = {'model': EMBEDDING_MODEL, 'input': batch_items}
            res = requests.post('https://openrouter.ai/api/v1/embeddings', headers=headers, json=payload, timeout=25)
            if res.status_code == 200:
                data = res.json()
                embeddings.extend([item['embedding'] for item in data['data']])
                success = True
            else:
                key_idx += 1
                time.sleep(1)
    return np.array(embeddings, dtype=np.float32)

print('Cloud embedding engine ready!')

In [ ]:
# Cell 4: Extract Feature Sets
ast_vectors = []
for vec_str in df_filtered['buggy_ast_vector']:
    try:
        arr = json.loads(vec_str)
        ast_vectors.append(arr if len(arr) == 64 else [0.0]*64)
    except Exception:
        ast_vectors.append([0.0]*64)
X_ast = np.array(ast_vectors, dtype=np.float32)

semantic_texts = []
for idx, row in df_filtered.iterrows():
    c_msg = str(row.get('original_commit_message', ''))
    b_desc = str(row.get('bug_description', ''))
    r_cause = str(row.get('root_cause', ''))
    f_desc = str(row.get('fix_description', ''))
    semantic_texts.append(f'COMMIT: {c_msg}\nBUG: {b_desc}\nROOT CAUSE: {r_cause}\nFIX: {f_desc}')

X_semantic = fetch_cloud_embeddings(semantic_texts)
X_hybrid = np.hstack([X_ast, X_semantic])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_filtered['bug_type'])
target_names = list(label_encoder.classes_)

print(f'Feature Set 1 (AST)      : {X_ast.shape}')
print(f'Feature Set 2 (Semantic) : {X_semantic.shape}')
print(f'Feature Set 3 (Hybrid)   : {X_hybrid.shape}')

In [ ]:
# Cell 5: Direct Bug Checker Engine
from predict_bug import check_bug
print('Direct Bug Checker Engine Ready!')

--- 
## 🔍 TEST ANY CODE BELOW (BUG OR NO BUG)
Paste **ANY** code in `code_input` below to see **`⚠️ BUG`** or **`✅ NO BUG`**!

In [ ]:
# Cell 6: Direct Bug Check Playground
code_input = """
const name = user.profile.name;
"""

result = check_bug(code_input)